# Full alphaP-WNS Drift Prior With Soft Topology Shrinkage on GOLDEN

This notebook prototypes the full Gillis-Sharma / alphaP-WNS construction on the full `GOLDEN` dynamic endogenous drift block and applies a strong Normal shrinkage factor to drift entries that the topology marks as absent. This is a prior-level benchmark only: no pipeline evals or inference runs are triggered.

In [1]:

from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
from scipy.linalg import expm


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "apps" / "data-pipeline").exists() and (current / "data").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate repository root from notebook working directory")


REPO_ROOT = find_repo_root(Path.cwd())
GOLDEN_RUN = REPO_ROOT / "data" / ".private" / "GOLDEN" / "run"
STAGE1B_PATH = GOLDEN_RUN / "stage-1b.json"
MEGAPROMPT_PATH = GOLDEN_RUN / "stage-4-megaprompt.json"

stage1b = json.loads(STAGE1B_PATH.read_text())
megaprompt = json.loads(MEGAPROMPT_PATH.read_text())
accepted = megaprompt["accepted"]
causal_spec = stage1b["causal_spec"]
resolved_priors = {
    prior["parameter"]: prior
    for prior in accepted["resolved_priors"]
    if prior is not None
}

all_state_names = list(causal_spec["estimation"]["state_order"])
constructs = {construct["name"]: construct for construct in causal_spec["latent"]["constructs"]}
latent_names = [
    name
    for name in all_state_names
    if constructs[name].get("role") == "endogenous"
    and constructs[name].get("temporal_status") == "time_varying"
]
n_latent = len(latent_names)
latent_index = {name: idx for idx, name in enumerate(latent_names)}

drift_mask = np.eye(n_latent, dtype=bool)
for edge in causal_spec["latent"]["edges"]:
    cause = edge["cause"]
    effect = edge["effect"]
    parameter = f"beta_{cause}_{effect}"
    if cause in latent_index and effect in latent_index and parameter in resolved_priors:
        drift_mask[latent_index[effect], latent_index[cause]] = True

diag_mask = np.eye(n_latent, dtype=bool)
allowed_offdiag_mask = drift_mask & ~diag_mask
structural_zero_mask = (~drift_mask) & ~diag_mask
allowed_positions = [
    (row, col)
    for row in range(n_latent)
    for col in range(n_latent)
    if row != col and drift_mask[row, col]
]


def duration_to_days(value: str) -> float:
    if value.endswith("d"):
        return float(value[:-1])
    if value.endswith("h"):
        return float(value[:-1]) / 24.0
    raise ValueError(f"Unsupported model clock {value!r}")


model_dt_days = duration_to_days(causal_spec["measurement"]["model_clock"])

print(f"Golden run: {GOLDEN_RUN.relative_to(REPO_ROOT)}")
print(f"Dynamic drift block ({n_latent} states): {', '.join(latent_names)}")
print("Excluded retained states:")
for name in all_state_names:
    if name not in latent_index:
        construct = constructs[name]
        print(f"  {name}: {construct.get('role')} / {construct.get('temporal_status')}")
print(f"Model interval: {model_dt_days:g} day")
print(f"Allowed off-diagonal dynamic drift entries: {int(allowed_offdiag_mask.sum())}")
print(f"Structural-zero off-diagonal dynamic entries: {int(structural_zero_mask.sum())}")
print("Allowed dynamic topology edges:")
for row, col in allowed_positions:
    print(f"  {latent_names[col]} -> {latent_names[row]}")


Golden run: data/.private/GOLDEN/run
Dynamic drift block (9 states): sleep_quality, sleep_duration, screen_time, evening_screen_use, screen_content_type, social_media_use, stress, mental_health, bedtime_delay
Excluded retained states:
  chronotype: exogenous / time_invariant
Model interval: 1 day
Allowed off-diagonal dynamic drift entries: 11
Structural-zero off-diagonal dynamic entries: 61
Allowed dynamic topology edges:
  sleep_duration -> sleep_quality
  stress -> sleep_quality
  mental_health -> sleep_quality
  bedtime_delay -> sleep_duration
  stress -> screen_time
  mental_health -> screen_time
  screen_time -> evening_screen_use
  screen_content_type -> social_media_use
  mental_health -> stress
  stress -> mental_health
  evening_screen_use -> bedtime_delay


## Construction

Draw `P^{-1}` as a positive-definite matrix, draw a skew-symmetric `S`, draw `alpha > 0`, and assemble `A = -0.5 P^{-1}(alpha P + S)`. Stability is guaranteed by construction. Topology is not hard-coded into the support; instead, candidate draws are reweighted by a tight Normal density on absent topology entries of `A`. The weighted-resampling step below is a notebook approximation to that soft prior.

In [2]:

def sample_prior_1d(prior: dict, rng: np.random.Generator, n_draws: int) -> np.ndarray:
    family = prior["distribution"]
    params = prior["params"]
    if family == "Beta":
        return rng.beta(params["alpha"], params["beta"], size=n_draws)
    if family == "Uniform":
        return rng.uniform(params["lower"], params["upper"], size=n_draws)
    if family == "Normal":
        return rng.normal(params["mu"], params["sigma"], size=n_draws)
    raise ValueError(f"Unsupported drift prior family {family!r}")


def sample_current_scalar_prior(rng: np.random.Generator, n_draws: int) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    for idx, name in enumerate(latent_names):
        rho = sample_prior_1d(resolved_priors[f"rho_{name}"], rng, n_draws)
        rho = np.clip(rho, 1e-8, 1.0 - 1e-8)
        draws[:, idx, idx] = np.log(rho) / model_dt_days
    for row, col in allowed_positions:
        beta = sample_prior_1d(resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"], rng, n_draws)
        draws[:, row, col] = beta / model_dt_days
    return draws


def interval_response_samples(draws: np.ndarray, dt_days: float, max_draws: int = 1000) -> np.ndarray:
    n = min(max_draws, draws.shape[0])
    return np.stack([expm(draws[i] * dt_days) for i in range(n)], axis=0)


def summarize_drift_samples(name: str, draws: np.ndarray, elapsed_seconds: float) -> dict[str, float | str]:
    eigvals = np.linalg.eigvals(draws)
    max_real = eigvals.real.max(axis=1)
    zero_abs = np.abs(draws[:, structural_zero_mask])
    allowed_abs = np.abs(draws[:, allowed_offdiag_mask])
    response = interval_response_samples(draws, model_dt_days)
    absent_response_abs = np.abs(response[:, structural_zero_mask])
    allowed_response_abs = np.abs(response[:, allowed_offdiag_mask])
    return {
        "name": name,
        "draws": draws.shape[0],
        "stable_rate": float(np.mean(max_real < 0.0)),
        "margin_q05": float(np.quantile(-max_real, 0.05)),
        "diag_mean": float(np.mean(np.diagonal(draws, axis1=1, axis2=2))),
        "absent_a_q90": float(np.quantile(zero_abs, 0.90)),
        "absent_a_max": float(np.max(zero_abs)),
        "allowed_a_q90": float(np.quantile(allowed_abs, 0.90)),
        "absent_response_q90": float(np.quantile(absent_response_abs, 0.90)),
        "allowed_response_q90": float(np.quantile(allowed_response_abs, 0.90)),
        "seconds": float(elapsed_seconds),
    }


def print_summary_table(rows: list[dict[str, float | str]]) -> None:
    columns = [
        ("name", "prior"),
        ("draws", "draws"),
        ("stable_rate", "stable"),
        ("margin_q05", "margin q05"),
        ("diag_mean", "diag mean"),
        ("absent_a_q90", "absent |A| q90"),
        ("absent_a_max", "absent |A| max"),
        ("allowed_a_q90", "allowed |A| q90"),
        ("absent_response_q90", "absent |exp(AΔ)| q90"),
        ("allowed_response_q90", "allowed |exp(AΔ)| q90"),
        ("seconds", "seconds"),
    ]
    widths = []
    for key, label in columns:
        values = [label]
        for row in rows:
            value = row[key]
            values.append(str(value) if isinstance(value, str) else f"{value:.4g}")
        widths.append(max(len(v) for v in values))
    header = "  ".join(label.ljust(width) for (_, label), width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in rows:
        parts = []
        for (key, _label), width in zip(columns, widths):
            value = row[key]
            text = str(value) if isinstance(value, str) else f"{value:.4g}"
            parts.append(text.ljust(width))
        print("  ".join(parts))


In [3]:

def sample_alpha_p_wns_candidates(
    rng: np.random.Generator,
    n_draws: int,
    *,
    skew_scale: float = 0.05,
    wishart_df_extra: int = 30,
    wishart_mix: float = 0.10,
    alpha_shape: float = 16.0,
) -> np.ndarray:
    df = n_latent + wishart_df_extra
    wishart_factors = rng.standard_normal((n_draws, df, n_latent))
    wishart = np.einsum("bki,bkj->bij", wishart_factors, wishart_factors) / df
    p_inv = (1.0 - wishart_mix) * np.eye(n_latent) + wishart_mix * wishart
    p_inv = p_inv + 1e-4 * np.eye(n_latent)
    p = np.linalg.inv(p_inv)

    skew = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    lower = np.tril_indices(n_latent, k=-1)
    skew_values = rng.normal(0.0, skew_scale, size=(n_draws, len(lower[0])))
    skew[:, lower[0], lower[1]] = skew_values
    skew[:, lower[1], lower[0]] = -skew_values

    scalar_reference = sample_current_scalar_prior(rng, min(n_draws, 2000))
    base_decay = float(np.median(-np.diagonal(scalar_reference, axis1=1, axis2=2)))
    alpha_mean = 2.0 * base_decay
    alpha = rng.gamma(shape=alpha_shape, scale=alpha_mean / alpha_shape, size=n_draws)
    q_plus_s = alpha[:, None, None] * p + skew
    return -0.5 * np.einsum("bij,bjk->bik", p_inv, q_plus_s)


def topology_shrinkage_weights(draws: np.ndarray, *, zero_sigma: float) -> tuple[np.ndarray, float]:
    absent_entries = draws[:, structural_zero_mask]
    logw = -0.5 * np.sum((absent_entries / zero_sigma) ** 2, axis=1)
    logw = logw - float(np.max(logw))
    weights = np.exp(logw)
    weights = weights / float(np.sum(weights))
    ess = 1.0 / float(np.sum(weights**2))
    return weights, ess


def resample_with_weights(
    rng: np.random.Generator,
    draws: np.ndarray,
    weights: np.ndarray,
    n_resampled: int,
) -> np.ndarray:
    indices = rng.choice(draws.shape[0], size=n_resampled, replace=True, p=weights)
    return draws[indices]


## Benchmark

In [4]:

N_BASELINE = 5000
N_CANDIDATES = 30000
N_RESAMPLED = 5000
ZERO_SIGMA = 0.05
rng = np.random.default_rng(20260428)

start = time.perf_counter()
scalar_draws = sample_current_scalar_prior(rng, N_BASELINE)
scalar_elapsed = time.perf_counter() - start

start = time.perf_counter()
wns_candidates = sample_alpha_p_wns_candidates(rng, N_CANDIDATES)
wns_candidate_elapsed = time.perf_counter() - start

weights, ess = topology_shrinkage_weights(wns_candidates, zero_sigma=ZERO_SIGMA)
start = time.perf_counter()
wns_shrunk = resample_with_weights(rng, wns_candidates, weights, N_RESAMPLED)
wns_resample_elapsed = time.perf_counter() - start

rows = [
    summarize_drift_samples("accepted scalar prior", scalar_draws, scalar_elapsed),
    summarize_drift_samples("raw alphaP-WNS", wns_candidates[:N_RESAMPLED], wns_candidate_elapsed),
    summarize_drift_samples(
        "alphaP-WNS + zero shrinkage",
        wns_shrunk,
        wns_candidate_elapsed + wns_resample_elapsed,
    ),
]
print_summary_table(rows)
print(f"Shrinkage zero_sigma: {ZERO_SIGMA:g}")
print(f"Candidate draws: {N_CANDIDATES}; resampled draws: {N_RESAMPLED}; ESS: {ess:.1f}")
print(f"ESS ratio: {ess / N_CANDIDATES:.4f}")


prior                        draws  stable  margin q05  diag mean  absent |A| q90  absent |A| max  allowed |A| q90  absent |exp(AΔ)| q90  allowed |exp(AΔ)| q90  seconds 
---------------------------  -----  ------  ----------  ---------  --------------  --------------  ---------------  --------------------  ---------------------  --------
accepted scalar prior        5000   1       3.569       -4.741     0               0               1.5              0.003515              0.01414                0.001451
raw alphaP-WNS               5000   1       2.906       -4.605     0.04107         0.1117          0.04095          0.0008567             0.0008475              0.2217  
alphaP-WNS + zero shrinkage  5000   1       2.909       -4.571     0.03387         0.1017          0.0374           0.0007377             0.0008118              0.2233  
Shrinkage zero_sigma: 0.05
Candidate draws: 30000; resampled draws: 5000; ESS: 5036.4
ESS ratio: 0.1679


## Shrinkage Diagnostics

In [5]:

absent_raw = np.abs(wns_candidates[:, structural_zero_mask]).ravel()
absent_shrunk = np.abs(wns_shrunk[:, structural_zero_mask]).ravel()
allowed_raw = np.abs(wns_candidates[:, allowed_offdiag_mask]).ravel()
allowed_shrunk = np.abs(wns_shrunk[:, allowed_offdiag_mask]).ravel()

for label, values in [
    ("raw absent entries", absent_raw),
    ("shrunk absent entries", absent_shrunk),
    ("raw allowed entries", allowed_raw),
    ("shrunk allowed entries", allowed_shrunk),
]:
    print(
        f"{label:<24s} median={np.median(values):.4f} "
        f"q90={np.quantile(values, 0.90):.4f} q99={np.quantile(values, 0.99):.4f}"
    )


raw absent entries       median=0.0168 q90=0.0412 q99=0.0645
shrunk absent entries    median=0.0139 q90=0.0339 q99=0.0534
raw allowed entries      median=0.0168 q90=0.0411 q99=0.0645
shrunk allowed entries   median=0.0152 q90=0.0374 q99=0.0589


## Reading

The full alphaP-WNS construction gives the cleanest stability geometry and keeps the scalar decay knob separated from the skew/rotation block. The topology is only approximate: the shrinkage factor lowers absent-edge drift mass but does not make absent effects impossible. This uncalibrated WNS prior also does not recover the accepted GOLDEN effect scale by itself; a production version would need semantic prior-predictive fitting for allowed interval effects in addition to absent-edge shrinkage.